# 05 — Extension: Diverse Members + Stacking

**Where we stand:** LB 0.94946 (blend of LGBM + CatBoost), spec target > 0.95, top ≈ 0.953. Milestone 4's blend gained little because its members agreed too much (error correlation 0.90). The lever this notebook pulls is **diversity**: add members that make *different mistakes*, then upgrade the combiner from a weight grid to a learned meta-model (stacking).

Two new members:

- **HistGradientBoosting** — sklearn's own GBM. Same family as LightGBM in spirit, but an independent implementation (different binning, different categorical treatment, different regularization defaults) — cheap to add and historically a useful decorrelator (the top public kernels for this competition stack exactly these four tree libraries).
- **An MLP** — a genuinely different function class: smooth decision boundaries where trees make axis-aligned cuts. Neural nets are usually *weaker* than GBMs on tabular data; the bet is not that it wins but that its errors are uncorrelated enough for the combiner to extract a few points from it. (Plan note: PyTorch has no wheels for this machine — Intel Mac + Python 3.13 — so we use sklearn's `MLPClassifier` and accept its constraints.)

The MLP also finally cashes in notebook 03's E2 lesson: **NNs can't eat NaN**, so we must impute — and E2 taught us imputation is only safe when paired with missingness flags.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

TARGET = 'health_condition'
NUM = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
       'step_count', 'exercise_duration', 'water_intake']
CAT = ['diet_type', 'stress_level', 'sleep_quality',
       'physical_activity_level', 'smoking_alcohol', 'gender']
FEATURES = NUM + CAT
SEED = 42
CLASSES = ['at-risk', 'fit', 'unhealthy']
y = train[TARGET].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(train[FEATURES], y))
fold_id = np.load('../artifacts/fold_id.npy')
for k, (tr, va) in enumerate(FOLDS):
    assert (fold_id[va] == k).all()

def cached_cv(name, fit_fn):
    op, tp = f'../artifacts/{name}_oof.npy', f'../artifacts/{name}_testp.npy'
    if os.path.exists(op):
        oof, testp = np.load(op), np.load(tp)
        print(f"{name}: loaded from cache")
    else:
        oof, testp = fit_fn()
        np.save(op, oof); np.save(tp, testp)
    print(f"{name}: OOF {balanced_accuracy_score(y, oof.argmax(1)):.5f}")
    return oof, testp

def tune_thresholds(oof_probs, y):
    def score(a, b):
        return balanced_accuracy_score(y, (oof_probs * np.array([1.0, a, b])).argmax(1))
    best = (1.0, 1.0, score(1.0, 1.0))
    for a in np.arange(0.5, 2.01, 0.05):
        for b in np.arange(0.5, 2.01, 0.05):
            s = score(a, b)
            if s > best[2]: best = (a, b, s)
    a0, b0, _ = best
    for a in np.arange(a0 - 0.06, a0 + 0.06, 0.01):
        for b in np.arange(b0 - 0.06, b0 + 0.06, 0.01):
            s = score(a, b)
            if s > best[2]: best = (a, b, s)
    return best

# Existing members from notebook 04's cache
oof_lgbm = np.mean([np.load(f'../artifacts/{n}_oof.npy')
                    for n in ('lgbm_tuned', 'lgbm_seed1', 'lgbm_seed2')], axis=0)
testp_lgbm = np.mean([np.load(f'../artifacts/{n}_testp.npy')
                      for n in ('lgbm_tuned', 'lgbm_seed1', 'lgbm_seed2')], axis=0)
oof_cat, testp_cat = np.load('../artifacts/cat_oof.npy'), np.load('../artifacts/cat_testp.npy')
print(f"lgbm seed-avg OOF {balanced_accuracy_score(y, oof_lgbm.argmax(1)):.5f}, "
      f"catboost OOF {balanced_accuracy_score(y, oof_cat.argmax(1)):.5f}")

lgbm seed-avg OOF 0.94831, catboost OOF 0.94923


## E5a — HistGradientBoosting

Near-defaults, deliberately: milestone 4 taught us tuning has little to give here. Native NaN handling and native categorical support (pandas `category` dtype), `class_weight='balanced'` — the same alignment lever as always. `max_iter=1000` with internal early stopping on a validation split.

In [2]:
from sklearn.ensemble import HistGradientBoostingClassifier

Xh = train[FEATURES].copy(); Xh_test = test[FEATURES].copy()
for c in CAT:
    Xh[c] = Xh[c].astype('category')
    Xh_test[c] = Xh_test[c].astype(pd.CategoricalDtype(Xh[c].cat.categories))

def fit_hgb():
    oof = np.zeros((len(Xh), 3)); testp = np.zeros((len(Xh_test), 3))
    for k, (tr, va) in enumerate(FOLDS):
        model = HistGradientBoostingClassifier(
            max_iter=1000, learning_rate=0.05, categorical_features='from_dtype',
            class_weight='balanced', early_stopping=True, n_iter_no_change=50,
            validation_fraction=0.1, random_state=SEED)
        model.fit(Xh.iloc[tr], y[tr])
        oof[va] = model.predict_proba(Xh.iloc[va])
        testp += model.predict_proba(Xh_test) / len(FOLDS)
    return oof, testp

oof_hgb, testp_hgb = cached_cv('hgb', fit_hgb)

hgb: OOF 0.94956


**Verdict: 0.94956 argmax — the best single model in the project**, from near-default parameters, again. Two notebooks of evidence now agree that this dataset rewards *implementation diversity* over parameter search: LightGBM 0.94894, CatBoost 0.94923, HGB 0.94956 — three GBM implementations, three scores, zero tuning between them. The differences come from what each library does differently by default (binning, categorical treatment, regularization), i.e., exactly the decorrelation we came for.

## E5b — MLP, with the preprocessing NNs demand

Everything trees shrugged at now matters:

- **NaN:** impute (median) — but per E2, *always* alongside missingness flags, or we destroy the `bmi` signal.
- **Scale:** standardize numerics — gradient descent on features spanning [1.86, 3580] vs [0, 3] is otherwise dominated by the large-scale features.
- **Categoricals:** one-hot — at 3–4 levels each this is cheap and exact; embeddings earn their complexity only at high cardinality.
- **In-fold discipline:** imputation medians and scaler statistics are fit on each fold's *training* part only. For medians over 552k rows the leakage from doing it globally is negligible (E2's aside), but here it costs nothing to do it right, so we do.
- **Class balance:** `MLPClassifier` supports neither class weights nor sample weights, so we reach for the third tool in the imbalance toolbox: **undersample the majority** in each training fold (`at-risk` capped at 3× the largest minority). Costs majority-class data we can afford (at-risk is the easy class); buys a training distribution the plain log-loss can serve. Thresholds afterwards, as always.

In [3]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

def prep_fold(tr_df, other_dfs):
    """Fit impute/scale/one-hot stats on tr_df only; apply to all frames."""
    med = tr_df[NUM].median()
    def transform(df):
        out = [df[NUM].fillna(med), df[NUM].isna().astype('int8').add_suffix('_miss'),
               df[CAT].isna().astype('int8').add_suffix('_miss')]
        for c in CAT:
            d = pd.get_dummies(df[c], prefix=c, dtype='int8')
            out.append(d)
        return pd.concat(out, axis=1)
    frames = [transform(tr_df)] + [transform(d) for d in other_dfs]
    cols = frames[0].columns
    frames = [f.reindex(columns=cols, fill_value=0) for f in frames]
    scaler = StandardScaler().fit(frames[0])
    return [scaler.transform(f).astype('float32') for f in frames]

def undersample(tr_idx, rng):
    counts = np.bincount(y[tr_idx]); cap = 3 * counts[1:].max()
    keep = []
    for cls in range(3):
        idx = tr_idx[y[tr_idx] == cls]
        if len(idx) > cap:
            idx = rng.choice(idx, cap, replace=False)
        keep.append(idx)
    return np.sort(np.concatenate(keep))

def fit_mlp():
    rng = np.random.default_rng(SEED)
    oof = np.zeros((len(train), 3)); testp = np.zeros((len(test), 3))
    for k, (tr, va) in enumerate(FOLDS):
        tr_u = undersample(tr, rng)
        Xtr, Xva, Xte = prep_fold(train[FEATURES].iloc[tr_u],
                                  [train[FEATURES].iloc[va], test[FEATURES]])
        model = MLPClassifier(hidden_layer_sizes=(256, 128), activation='relu',
                              alpha=1e-4, batch_size=1024, learning_rate_init=1e-3,
                              max_iter=60, early_stopping=True, n_iter_no_change=5,
                              validation_fraction=0.1, random_state=SEED)
        model.fit(Xtr, y[tr_u])
        oof[va] = model.predict_proba(Xva)
        testp += model.predict_proba(Xte) / len(FOLDS)
        print(f"  fold {k}: {balanced_accuracy_score(y[va], oof[va].argmax(1)):.5f} "
              f"({model.n_iter_} epochs)")
    return oof, testp

oof_mlp, testp_mlp = cached_cv('mlp', fit_mlp)
a_m, b_m, mlp_thresh = tune_thresholds(oof_mlp, y)
print(f"mlp + thresholds: {mlp_thresh:.5f}")

  fold 0: 0.93434 (18 epochs)


  fold 1: 0.93436 (25 epochs)


  fold 2: 0.93899 (22 epochs)


  fold 3: 0.93593 (16 epochs)


  fold 4: 0.93736 (24 epochs)
mlp: OOF 0.93620


mlp + thresholds: 0.94465


**Verdict: 0.93620 argmax → 0.94465 thresholded — the weakest member, as expected.** Three observations worth the compute:

- **The threshold gain (+0.0085) is the largest of any model** — no surprise: this is the only model trained on a plain unweighted loss (undersampling rebalanced the data only partially, at 3:1), so its decision boundary had the most mis-centering for thresholds to fix. The imbalance toolbox — weights, resampling, thresholds — is genuinely interchangeable at the margin.
- **It is by far the most *diverse* member** (error correlation 0.78–0.81 vs 0.90–0.955 among the trees). The diversity bet was half right.
- **Quality floors matter more.** The blend gave it weight 0: a 0.005+ quality gap swamps a 0.12 correlation advantage. Diversity only pays above a competence threshold — sklearn's MLP, without embeddings, LR schedules, or serious tuning (the parts a real NN framework would provide), doesn't reach it on this data.

## Combining: blend grid vs stacking

**Error correlations first** — the number that predicts whether any of this can work.

Then two combiners on the same four members:

1. **Weight grid** (as milestone 4): exhaustive simplex search, step 0.05, then thresholds. Simple, robust, only 3 free parameters — but it can only *scale* each model's vote.
2. **Stacking:** a multinomial logistic regression trained on the concatenated OOF probabilities (12 columns → 3 classes). A learned combiner can do what fixed weights cannot: trust CatBoost for `unhealthy` but LightGBM for `fit`, or exploit *disagreement patterns* between members as signal. The OOF matrix makes this legitimate — every row is an honest prediction — and we validate the meta-model with its own 5-fold CV on those OOF features (same folds again), so the stacker's score is itself out-of-fold. With `class_weight='balanced'` the stacker's linear layer also absorbs the threshold job (a per-class intercept is exactly a per-class multiplier in log-space), though we still check whether explicit thresholds add anything on top.

In [4]:
members = {'lgbm': (oof_lgbm, testp_lgbm), 'cat': (oof_cat, testp_cat),
           'hgb': (oof_hgb, testp_hgb), 'mlp': (oof_mlp, testp_mlp)}
err = pd.DataFrame({n: (o.argmax(1) != y).astype(int) for n, (o, _) in members.items()})
print("error correlation:\n", err.corr().round(3))

error correlation:
        lgbm    cat    hgb    mlp
lgbm  1.000  0.902  0.906  0.806
cat   0.902  1.000  0.955  0.784
hgb   0.906  0.955  1.000  0.782
mlp   0.806  0.784  0.782  1.000


In [5]:
from itertools import product

names = list(members)
grid = np.arange(0, 1.0001, 0.05)
best = (None, -1)
for ws in product(grid, repeat=len(names) - 1):
    if sum(ws) > 1: continue
    w = np.array([*ws, 1 - sum(ws)])
    oof_b = sum(w[i] * members[n][0] for i, n in enumerate(names))
    s = balanced_accuracy_score(y, oof_b.argmax(1))
    if s > best[1]: best = (w, s)
W, blend_argmax = best
print(f"weights {dict(zip(names, W.round(2)))}: blended OOF (argmax) {blend_argmax:.5f}")
oof_blend = sum(W[i] * members[n][0] for i, n in enumerate(names))
testp_blend = sum(W[i] * members[n][1] for i, n in enumerate(names))
a_b, b_b, blend_cv = tune_thresholds(oof_blend, y)
print(f"blend + thresholds (fit={a_b:.2f}, unh={b_b:.2f}): {blend_cv:.5f}")

weights {'lgbm': np.float64(0.1), 'cat': np.float64(0.2), 'hgb': np.float64(0.7), 'mlp': np.float64(0.0)}: blended OOF (argmax) 0.94986


blend + thresholds (fit=1.11, unh=1.08): 0.94990


In [6]:
from sklearn.linear_model import LogisticRegression

Z = np.hstack([members[n][0] for n in names])
Z_test = np.hstack([members[n][1] for n in names])

oof_stack = np.zeros((len(train), 3))
for k, (tr, va) in enumerate(FOLDS):
    meta = LogisticRegression(max_iter=2000, class_weight='balanced', C=1.0)
    meta.fit(Z[tr], y[tr])
    oof_stack[va] = meta.predict_proba(Z[va])
stack_argmax = balanced_accuracy_score(y, oof_stack.argmax(1))
a_s, b_s, stack_cv = tune_thresholds(oof_stack, y)
print(f"stack (argmax): {stack_argmax:.5f}, + thresholds: {stack_cv:.5f}")

meta_full = LogisticRegression(max_iter=2000, class_weight='balanced', C=1.0)
meta_full.fit(Z, y)
testp_stack = meta_full.predict_proba(Z_test)

stack (argmax): 0.94926, + thresholds: 0.94985


**Verdict: blend 0.94990, stack 0.94985 — a statistical tie, blend narrowly ahead.**

The learned combiner earned nothing over three fixed weights plus two threshold multipliers. That's the expected result *for this configuration*: with few members whose probabilities are highly correlated and individually well-calibrated for the task, the optimal combination genuinely is close to a weighted average — there's no per-class trust structure or disagreement pattern left to learn. (The stacker's balanced class weights did absorb the threshold job, as predicted: its argmax score (0.94926) is only 0.0006 behind its thresholded one.) Stacking earns its keep with many heterogeneous members — 10+ models of different families, where per-class trust assignment has real signal to find. Worth knowing both tools; worth knowing when each is the right one.

One more instructive weight vector: the grid put **0.7 on HGB despite its 0.955 correlation with CatBoost** — blend weights track quality first, diversity second. The textbook picture (weights ∝ decorrelation) applies only among near-equals.

## Submission

In [7]:
def make_submission(test_probs, mult, name):
    labels = [CLASSES[i] for i in (test_probs * np.array([1.0, *mult])).argmax(1)]
    sub = pd.DataFrame({'id': test['id'], TARGET: labels})
    sub.to_csv(f'../submissions/{name}.csv', index=False)
    print(f"{name}: {dict(sub[TARGET].value_counts(normalize=True).round(4))}")

if stack_cv >= blend_cv:
    print(f"stacker wins on CV ({stack_cv:.5f} vs {blend_cv:.5f})")
    make_submission(testp_stack, (a_s, b_s), '05-stack')
else:
    print(f"blend wins on CV ({blend_cv:.5f} vs {stack_cv:.5f})")
    make_submission(testp_blend, (a_b, b_b), '05-stack')

summary = pd.DataFrame({'OOF bal. acc. (thresholded where applicable)': {
    'nb04 blend + thresholds (final A)': 0.94968,
    'hgb (argmax)': balanced_accuracy_score(y, oof_hgb.argmax(1)),
    'mlp + thresholds': mlp_thresh,
    '4-way blend + thresholds': blend_cv,
    'stack (argmax)': stack_argmax,
    'stack + thresholds': stack_cv}}).round(5)
summary

blend wins on CV (0.94990 vs 0.94985)


05-stack: {'at-risk': np.float64(0.8098), 'unhealthy': np.float64(0.1164), 'fit': np.float64(0.0738)}


,OOF bal. acc. (thresholded where applicable)
nb04 blend + thresholds (final A),0.94968
hgb (argmax),0.94956
mlp + thresholds,0.94465
4-way blend + thresholds,0.94990
stack (argmax),0.94926
stack + thresholds,0.94985


## Result

| submission | CV | public LB | CV − LB |
|---|---|---|---|
| nb04 blend (final A) | 0.94968 | 0.94946 | +0.0002 |
| **nb05 4-way blend + thresholds** | **0.94990** | **0.94978** | +0.0001 |

New best public score, and the CV↔LB agreement stays inside ±0.001 — five submissions of evidence that the harness measures what it claims to.

Kaggle's auto-selection now picks 05-stack (0.94978) and 04-final-blend (0.94946) as the two finals — a reasonable pair: different member sets, different combiner fits.

## Extension recap

The spec target was >0.95; we close at **0.94978 public / 0.94990 CV** — a hair short on the public split, within our measured noise of the target on CV. The extension's yield: +0.0032 LB over milestone 4, from one cheap idea (a third GBM implementation) and one confirmed-negative (sklearn's MLP can't reach the quality floor on this hardware/framework budget). What would plausibly cross 0.953-territory: a serious NN (RealMLP/TabM with embeddings and schedules — needs PyTorch, unavailable on this machine), 10+ member stacks, and multi-seed everything — all steeply diminishing returns per engineering hour. The learning goals — EDA, CV trust, missing-data reasoning, GBM mastery, decision-rule alignment, ensembling — are fully banked.